In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/accident/test_metadata.csv
/kaggle/input/competitions/accident/videos/LY4ki_2dmwU_00.mp4
/kaggle/input/competitions/accident/videos/TAx0an0Jo84_00.mp4
/kaggle/input/competitions/accident/videos/dgWpyczEx7U_00.mp4
/kaggle/input/competitions/accident/videos/Nn10f6awuTk_00.mp4
/kaggle/input/competitions/accident/videos/seHQ5Y5g5Ns_00.mp4
/kaggle/input/competitions/accident/videos/wScVipt0pWk_10_00.mp4
/kaggle/input/competitions/accident/videos/trvzPZZfNPs_00.mp4
/kaggle/input/competitions/accident/videos/oPHbqlp1o5k_00.mp4
/kaggle/input/competitions/accident/videos/Yf9mgizvxiE_00.mp4
/kaggle/input/competitions/accident/videos/EN-JKUXkwPQ_7_00.mp4
/kaggle/input/competitions/accident/videos/-yuVlpyayOI_00.mp4
/kaggle/input/competitions/accident/videos/HgLFvVtCXRw_00.mp4
/kaggle/input/competitions/accident/videos/7NMgRqMghNs_00.mp4
/kaggle/input/competitions/accident/videos/fd3l6Z3OSXE_00.mp4
/kaggle/input/competitions/accident/videos/Qbgh3C8ew2o_00.mp4
/kaggle/inp

In [ ]:
# =========================================================
# ACCIDENT@CVPR — VideoMAE-v2 + TriDet + Trajectory Classification
# Pipeline:
#   1. GroundingDINO + ByteTrack at ~2 FPS → track proposals + candidate windows
#   2. VideoMAE-v2 Base → per-clip temporal features (ROI-focused, candidate-only)
#   3. TriDet-style temporal boundary refinement (trained on CARLA synthetic)
#   4. Trajectory-based collision type classification
# Output: /kaggle/working/submission.csv
# =========================================================

import os, sys, glob, math, warnings, json, gc
from itertools import combinations
from collections import defaultdict
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
warnings.filterwarnings("ignore")

# ── Paths ────────────────────────────────────────────────
COMP_ROOT      = "/kaggle/input/competitions/accident"
SIM_ROOT       = f"{COMP_ROOT}/sim_dataset"
LABELS_CSV     = f"{SIM_ROOT}/labels.csv"
TEST_VIDEO_DIR = f"{COMP_ROOT}/videos"
TEST_VIDEOS    = sorted(glob.glob(f"{TEST_VIDEO_DIR}/*.mp4"))
print("[OK] #test videos:", len(TEST_VIDEOS))
if len(TEST_VIDEOS) == 0:
    raise FileNotFoundError("No test videos found.")

# ── Install dependencies ────────────────────────────────
!pip -q install --upgrade pip
!pip -q install opencv-python pillow tqdm
!pip -q install transformers accelerate safetensors timm einops
!pip -q install supervision==0.22.0
!pip -q install scikit-learn

import torch
import torch.nn as nn
import torch.nn.functional as F
import supervision as sv
from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection,
    VideoMAEModel,
    VideoMAEImageProcessor,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("[OK] DEVICE:", DEVICE)

# ═══════════════════════════════════════════════════════════
# §1  GroundingDINO  (zero-shot vehicle detector)
# ═══════════════════════════════════════════════════════════
GROUNDING_ID     = "IDEA-Research/grounding-dino-base"
TEXT_PROMPT       = "car. bus. truck. motorcycle. bicycle. rickshaw. van. auto rickshaw."
BOX_THRESHOLD     = 0.28
TEXT_THRESHOLD    = 0.25
RESIZE_LONG_EDGE  = 640

processor_gd = AutoProcessor.from_pretrained(GROUNDING_ID)
detector_gd  = AutoModelForZeroShotObjectDetection.from_pretrained(GROUNDING_ID).to(DEVICE).eval()

def resize_keep_aspect(img, long_edge):
    h, w = img.shape[:2]
    scale = long_edge / max(h, w)
    if scale >= 1.0:
        return img, 1.0
    nh, nw = int(round(h * scale)), int(round(w * scale))
    return cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR), scale

@torch.no_grad()
def detect_objects(frame_rgb):
    fr, scale = resize_keep_aspect(frame_rgb, RESIZE_LONG_EDGE)
    inputs = processor_gd(images=fr, text=TEXT_PROMPT, return_tensors="pt").to(DEVICE)
    outputs = detector_gd(**inputs)
    target_sizes = torch.tensor([fr.shape[:2]], device=DEVICE)
    try:
        res = processor_gd.post_process_grounded_object_detection(
            outputs, inputs.input_ids, BOX_THRESHOLD, TEXT_THRESHOLD, target_sizes)[0]
    except TypeError:
        try:
            res = processor_gd.post_process_grounded_object_detection(
                outputs, inputs.input_ids, target_sizes, BOX_THRESHOLD, TEXT_THRESHOLD)[0]
        except TypeError:
            res = processor_gd.post_process_grounded_object_detection(
                outputs, inputs.input_ids, target_sizes)[0]
    if len(res.get("boxes", [])) == 0:
        return sv.Detections.empty()
    boxes  = res["boxes"].detach().cpu().numpy()
    scores = res["scores"].detach().cpu().numpy()
    keep = scores >= BOX_THRESHOLD
    boxes, scores = boxes[keep], scores[keep]
    if len(boxes) == 0:
        return sv.Detections.empty()
    if scale != 1.0:
        boxes = boxes / scale
    return sv.Detections(
        xyxy=boxes.astype(np.float32),
        confidence=scores.astype(np.float32),
        class_id=np.zeros(len(boxes), dtype=np.int32),
    )

# ═══════════════════════════════════════════════════════════
# §2  VideoMAE-v2  (temporal feature extractor)
# ═══════════════════════════════════════════════════════════
VMAE_ID = "MCG-NJU/videomae-base"
vmae_proc  = VideoMAEImageProcessor.from_pretrained(VMAE_ID)
vmae_model = VideoMAEModel.from_pretrained(VMAE_ID).to(DEVICE).eval()
VMAE_NUM_FRAMES = 16
VMAE_DIM = 768

@torch.no_grad()
def extract_vmae_features(clip_frames_rgb):
    """Extract a single mean-pooled feature vector from a 16-frame clip."""
    if len(clip_frames_rgb) < VMAE_NUM_FRAMES:
        while len(clip_frames_rgb) < VMAE_NUM_FRAMES:
            clip_frames_rgb = clip_frames_rgb + [clip_frames_rgb[-1]]
    elif len(clip_frames_rgb) > VMAE_NUM_FRAMES:
        idxs = np.linspace(0, len(clip_frames_rgb)-1, VMAE_NUM_FRAMES).astype(int)
        clip_frames_rgb = [clip_frames_rgb[i] for i in idxs]
    inputs = vmae_proc(clip_frames_rgb, return_tensors="pt").to(DEVICE)
    out = vmae_model(**inputs)
    feat = out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()
    return feat

# ═══════════════════════════════════════════════════════════
# §3  Tracking helpers
# ═══════════════════════════════════════════════════════════
def read_video_stream(path):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 1e-6:
        fps = 30.0
    return cap, float(fps)

def load_frames_limited(path, max_seconds=None):
    cap, fps = read_video_stream(path)
    frames, max_frames = [], int((max_seconds * fps) if max_seconds else 1e18)
    while True:
        ok, bgr = cap.read()
        if not ok or len(frames) >= max_frames:
            break
        frames.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames, fps

def center(bb):
    return np.array([(bb[0]+bb[2])/2, (bb[1]+bb[3])/2], dtype=np.float32)

def iou_xyxy(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0., ix2-ix1), max(0., iy2-iy1)
    inter = iw * ih
    aa = max(0., a[2]-a[0]) * max(0., a[3]-a[1])
    ab = max(0., b[2]-b[0]) * max(0., b[3]-b[1])
    return float(inter / (aa + ab - inter + 1e-9))

def compute_iou(b1, b2):
    return iou_xyxy(b1, b2)

def impact_point_from_boxes(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    if ix2 > ix1 and iy2 > iy1:
        return np.array([(ix1+ix2)/2, (iy1+iy2)/2], dtype=np.float32)
    ca, cb = center(a), center(b)
    pa = np.array([np.clip(cb[0], a[0], a[2]), np.clip(cb[1], a[1], a[3])], dtype=np.float32)
    pb = np.array([np.clip(ca[0], b[0], b[2]), np.clip(ca[1], b[1], b[3])], dtype=np.float32)
    return (pa + pb) / 2.0

def build_tracker(fps):
    return sv.ByteTrack(
        track_activation_threshold=0.25,
        lost_track_buffer=30,
        minimum_matching_threshold=0.8,
        frame_rate=int(round(fps)),
    )

# ── Run detection + tracking on a full video ─────────────
DETECT_FPS = 2.0          # detection rate (2 FPS = fast)
MAX_VIDEO_SEC = 20.0      # cap

def run_tracking(frames, fps, detect_fps=DETECT_FPS):
    stride = max(1, int(round(fps / detect_fps)))
    tracker = build_tracker(fps)
    det = {'frames': [], 'bboxes': [], 'track_ids': []}
    trajectories = defaultdict(lambda: {'frames': [], 'bboxes': [], 'velocities': []})

    for idx in range(0, len(frames), stride):
        d = detect_objects(frames[idx])
        d_tr = tracker.update_with_detections(d)
        if d_tr.tracker_id is None or len(d_tr) == 0:
            det['frames'].append(idx)
            det['bboxes'].append(np.empty((0,4), dtype=np.float32))
            det['track_ids'].append([])
            continue

        bbs  = d_tr.xyxy.astype(np.float32)
        tids = [int(t) for t in d_tr.tracker_id]

        det['frames'].append(idx)
        det['bboxes'].append(bbs)
        det['track_ids'].append(tids)

        for bb, tid in zip(bbs, tids):
            c = center(bb)
            traj = trajectories[tid]
            if traj['frames']:
                dt_frames = idx - traj['frames'][-1]
                dt_sec = dt_frames / fps if dt_frames > 0 else 1.0
                prev_c = center(traj['bboxes'][-1])
                vel = ((c - prev_c) / dt_sec).tolist()
            else:
                vel = [0.0, 0.0]
            traj['frames'].append(idx)
            traj['bboxes'].append(bb.copy())
            traj['velocities'].append(vel)

    return det, dict(trajectories)

# ── Interaction score for coarse candidate picking ───────
W_IOU = 1.5
W_CLOSING = 1.0
W_DECEL = 0.6

def compute_interaction_scores(det, trajectories, fps):
    scores = []
    for k, f in enumerate(det['frames']):
        bbs   = det['bboxes'][k]
        tids  = det['track_ids'][k]
        if len(tids) < 2:
            scores.append((f, 0.0, None))
            continue
        best_s, best_pair = 0.0, None
        for i, j in combinations(range(len(tids)), 2):
            if tids[i] == tids[j]:
                continue
            s_iou = iou_xyxy(bbs[i], bbs[j])
            s_close = 0.0
            if k > 0:
                prev_tids = det['track_ids'][k-1]
                prev_bbs  = det['bboxes'][k-1]
                if tids[i] in prev_tids and tids[j] in prev_tids:
                    pi = prev_tids.index(tids[i])
                    pj = prev_tids.index(tids[j])
                    d_prev = np.linalg.norm(center(prev_bbs[pi]) - center(prev_bbs[pj]))
                    d_now  = np.linalg.norm(center(bbs[i]) - center(bbs[j]))
                    s_close = max(0., d_prev - d_now)
            def _decel(tid):
                tr = trajectories.get(tid)
                if tr is None or len(tr['velocities']) < 2:
                    return 0.0
                sp_prev = np.linalg.norm(tr['velocities'][-2])
                sp_now  = np.linalg.norm(tr['velocities'][-1])
                return max(0., sp_prev - sp_now)
            s_decel = _decel(tids[i]) + _decel(tids[j])
            s = W_IOU * s_iou + W_CLOSING * s_close + W_DECEL * s_decel
            if s > best_s:
                best_s, best_pair = s, (tids[i], tids[j])
        scores.append((f, best_s, best_pair))
    return scores

def pick_top_candidates(scores, fps, n_candidates=3, min_gap_sec=2.0):
    sorted_sc = sorted(scores, key=lambda x: x[1], reverse=True)
    min_gap = int(min_gap_sec * fps)
    picks = []
    for f, s, pair in sorted_sc:
        if any(abs(f - pf) < min_gap for pf, _, _ in picks):
            continue
        picks.append((f, s, pair))
        if len(picks) >= n_candidates:
            break
    return picks

# ═══════════════════════════════════════════════════════════
# §4  TriDet-style temporal localizer  (lightweight)
# ═══════════════════════════════════════════════════════════
class TriDetHead(nn.Module):
    def __init__(self, feat_dim=VMAE_DIM, hidden=256, n_layers=3):
        super().__init__()
        layers = []
        in_d = feat_dim
        for _ in range(n_layers):
            layers.extend([
                nn.Conv1d(in_d, hidden, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
            ])
            in_d = hidden
        self.backbone = nn.Sequential(*layers)
        self.head_d1 = nn.Conv1d(hidden, 3, kernel_size=3, padding=1, dilation=1)
        self.head_d2 = nn.Conv1d(hidden, 3, kernel_size=3, padding=2, dilation=2)
        self.head_d4 = nn.Conv1d(hidden, 3, kernel_size=3, padding=4, dilation=4)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        h = self.backbone(x)
        o1 = self.head_d1(h)
        o2 = self.head_d2(h)
        o4 = self.head_d4(h)
        out = (o1 + o2 + o4) / 3.0
        out = out.permute(0, 2, 1)
        conf = out[:, :, 0].sigmoid()
        dt   = out[:, :, 1:].relu()
        return conf, dt

tridet = TriDetHead(VMAE_DIM, hidden=256, n_layers=3).to(DEVICE)

# ═══════════════════════════════════════════════════════════
# §5  Training on CARLA synthetic data
# ═══════════════════════════════════════════════════════════
TRAIN_SYNTH_MAX = 60
DELTA_SEC       = 0.75
CANDIDATE_WINDOW_SEC = 3.0   # ±3s around each candidate for feature extraction

def resolve_sim_video(rgb_path):
    p = os.path.join(SIM_ROOT, rgb_path)
    if os.path.exists(p):
        return p
    base = os.path.basename(rgb_path)
    cand = glob.glob(os.path.join(SIM_ROOT, "videos", "**", base), recursive=True)
    return cand[0] if cand else None

def build_feature_sequence(frames, fps, det, trajectories, clip_stride_sec=1.0,
                           candidate_frames=None):
    """
    Extract VideoMAE features ONLY around candidate windows (not full video).
    candidate_frames: list of frame indices to centre windows around.
    If None, falls back to scanning the whole video.
    """
    clip_stride = max(1, int(clip_stride_sec * fps))
    window = max(1, int(VMAE_NUM_FRAMES * (fps / 8.0)))
    H, W = frames[0].shape[:2]
    T = len(frames)

    frame_to_di = {f: i for i, f in enumerate(det['frames'])}

    # Build set of start positions to scan
    if candidate_frames is not None and len(candidate_frames) > 0:
        starts = set()
        radius = int(CANDIDATE_WINDOW_SEC * fps)
        for cf in candidate_frames:
            lo = max(0, cf - radius)
            hi = min(T - window, cf + radius)
            for s in range(lo, hi + 1, clip_stride):
                starts.add(s)
        starts = sorted(starts)
    else:
        starts = list(range(0, T - window + 1, clip_stride))

    feats, stamps = [], []
    for start in starts:
        end = start + window
        if end > T:
            continue
        mid = (start + end) // 2
        idxs = np.linspace(start, end-1, VMAE_NUM_FRAMES).astype(int)
        clip = [frames[i] for i in idxs]

        roi = None
        for det_f in det['frames']:
            if abs(det_f - mid) <= clip_stride:
                di = frame_to_di[det_f]
                if len(det['bboxes'][di]) >= 2:
                    all_bb = det['bboxes'][di]
                    x1 = max(0, int(all_bb[:, 0].min()) - 20)
                    y1 = max(0, int(all_bb[:, 1].min()) - 20)
                    x2 = min(W, int(all_bb[:, 2].max()) + 20)
                    y2 = min(H, int(all_bb[:, 3].max()) + 20)
                    if (x2 - x1) > 40 and (y2 - y1) > 40:
                        roi = (x1, y1, x2, y2)
                break

        if roi is not None:
            clip = [cv2.resize(f[roi[1]:roi[3], roi[0]:roi[2]], (224, 224)) for f in clip]
        else:
            clip = [cv2.resize(f, (224, 224)) for f in clip]

        feat = extract_vmae_features(clip)
        feats.append(feat)
        stamps.append(mid / fps)

    if not feats:
        idxs = np.linspace(0, T-1, VMAE_NUM_FRAMES).astype(int)
        clip = [cv2.resize(frames[i], (224, 224)) for i in idxs]
        feats.append(extract_vmae_features(clip))
        stamps.append(T / (2 * fps))

    return np.stack(feats, axis=0), np.array(stamps, dtype=np.float32)

def train_tridet():
    """Train the TriDet head on synthetic CARLA videos."""
    if not os.path.exists(LABELS_CSV):
        print("[WARN] No labels.csv found – skipping TriDet training")
        return False
    df = pd.read_csv(LABELS_CSV)
    need = ["rgb_path", "accident_frame", "width", "height"]
    for c in need:
        if c not in df.columns:
            print(f"[WARN] Column {c} missing – skipping training")
            return False

    df_train = df.sample(min(TRAIN_SYNTH_MAX, len(df)), random_state=42)

    optimizer = torch.optim.AdamW(tridet.parameters(), lr=1e-3, weight_decay=1e-4)
    tridet.train()
    total_loss = 0.0
    n_used = 0

    for _, row in tqdm(df_train.iterrows(), total=len(df_train), desc="TriDet training (synth)"):
        vpath = resolve_sim_video(str(row["rgb_path"]))
        if vpath is None:
            continue
        frames, fps = load_frames_limited(vpath, 15.0)
        if len(frames) < 16:
            continue

        acc_time = float(row["accident_frame"]) / fps
        det_data, traj_data = run_tracking(frames, fps, detect_fps=2.0)
        feats, stamps = build_feature_sequence(frames, fps, det_data, traj_data, clip_stride_sec=1.0)
        T = len(stamps)
        if T < 3:
            continue

        t_start = acc_time - DELTA_SEC
        t_end   = acc_time + DELTA_SEC
        target_conf = np.zeros(T, dtype=np.float32)
        target_dt   = np.zeros((T, 2), dtype=np.float32)
        for i, t in enumerate(stamps):
            if t_start <= t <= t_end:
                target_conf[i] = 1.0
            target_dt[i, 0] = max(0., t - t_start)
            target_dt[i, 1] = max(0., t_end - t)

        x = torch.tensor(feats, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        gt_c = torch.tensor(target_conf).unsqueeze(0).to(DEVICE)
        gt_d = torch.tensor(target_dt).unsqueeze(0).to(DEVICE)

        pred_c, pred_d = tridet(x)

        bce = F.binary_cross_entropy(pred_c, gt_c, reduction='none')
        pt = torch.where(gt_c == 1, pred_c, 1 - pred_c)
        focal = ((1 - pt) ** 2) * bce
        loss_conf = focal.mean()

        mask = gt_c.unsqueeze(-1).expand_as(gt_d)
        loss_reg = (F.l1_loss(pred_d, gt_d, reduction='none') * mask).sum() / (mask.sum() + 1e-6)

        loss = loss_conf + 1.0 * loss_reg
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_used += 1

    tridet.eval()
    if n_used > 0:
        print(f"[OK] TriDet trained on {n_used} synth videos, avg loss = {total_loss/n_used:.4f}")
    return n_used > 0

tridet_trained = train_tridet()

# ═══════════════════════════════════════════════════════════
# §6  Classification from trajectories (rule-based)
# ═══════════════════════════════════════════════════════════
SPEED_THRESHOLD = 3.0

def _get_acceleration_at_frame(track, frame):
    if track is None or len(track['velocities']) < 2:
        return 0.0
    for i, f in enumerate(track['frames']):
        if f >= frame and i >= 1:
            v_now  = np.array(track['velocities'][i])
            v_prev = np.array(track['velocities'][i-1])
            dt = (track['frames'][i] - track['frames'][i-1])
            if dt == 0:
                return 0.0
            return float(np.linalg.norm(v_now - v_prev) / dt)
    return 0.0

def find_colliding_pair(det, trajectories, acc_frame, fps):
    frame_to_idx = {f: i for i, f in enumerate(det['frames'])}
    window_frames = [f for f in det['frames'] if abs(f - acc_frame) <= int(fps * 1.0)]

    best_iou = 0.0
    best_pair = None
    for f in window_frames:
        fi = frame_to_idx[f]
        bboxes = det['bboxes'][fi]
        tids   = det['track_ids'][fi]
        if len(tids) < 2:
            continue
        for (i, b1), (j, b2) in combinations(enumerate(bboxes), 2):
            if i >= len(tids) or j >= len(tids) or tids[i] == tids[j]:
                continue
            iou = compute_iou(b1, b2)
            if iou > best_iou:
                best_iou = iou
                best_pair = (tids[i], tids[j])

    if best_iou > 0.01:
        return best_pair

    best_score = -1.0
    for f in window_frames:
        fi = frame_to_idx[f]
        bboxes = det['bboxes'][fi]
        tids   = det['track_ids'][fi]
        if len(tids) < 2:
            continue
        for (i, b1), (j, b2) in combinations(enumerate(bboxes), 2):
            if i >= len(tids) or j >= len(tids) or tids[i] == tids[j]:
                continue
            c1 = center(b1)
            c2 = center(b2)
            dist = float(np.linalg.norm(c1 - c2))
            proximity = 1.0 / (dist + 1.0)
            a1 = _get_acceleration_at_frame(trajectories.get(tids[i]), f)
            a2 = _get_acceleration_at_frame(trajectories.get(tids[j]), f)
            acc_score = a1 + a2
            score = proximity * 0.5 + (acc_score / (acc_score + 1e-8)) * 0.5
            if score > best_score:
                best_score = score
                best_pair = (tids[i], tids[j])
    return best_pair

def get_pre_collision_velocity(track, acc_frame, fps, lookback_seconds=1.5):
    lookback_frames = int(lookback_seconds * fps)
    relevant_vels = []
    for i, f in enumerate(track['frames'][:-1]):
        if (acc_frame - lookback_frames) <= f < acc_frame:
            if i < len(track['velocities']):
                relevant_vels.append(track['velocities'][i])
    if not relevant_vels:
        return track['velocities'][-1] if track['velocities'] else (0, 0)
    return tuple(np.mean(relevant_vels, axis=0))

def get_bbox_at_frame(track, target_frame):
    for i, f in enumerate(track['frames']):
        if f >= target_frame:
            return track['bboxes'][i]
    return track['bboxes'][-1] if track['bboxes'] else None

def classify_from_trajectories(det, trajectories, accident_time, fps):
    acc_frame = int(accident_time * fps)
    colliding_pair = find_colliding_pair(det, trajectories, acc_frame, fps)
    if colliding_pair is None:
        nearby_tracks = set()
        frame_to_idx = {f: i for i, f in enumerate(det['frames'])}
        for f in det['frames']:
            if abs(f - acc_frame) < 3 * fps:
                fi = frame_to_idx[f]
                nearby_tracks.update(det['track_ids'][fi])
        if len(nearby_tracks) <= 1:
            return "single"
        return "rear-end"

    tid1, tid2 = colliding_pair
    track1 = trajectories.get(tid1)
    track2 = trajectories.get(tid2)
    if track1 is None or track2 is None:
        return "single"

    vel1 = get_pre_collision_velocity(track1, acc_frame, fps, 1.5)
    vel2 = get_pre_collision_velocity(track2, acc_frame, fps, 1.5)
    speed1 = np.hypot(vel1[0], vel1[1])
    speed2 = np.hypot(vel2[0], vel2[1])

    if speed1 < SPEED_THRESHOLD and speed2 < SPEED_THRESHOLD:
        return "single"

    if speed1 < SPEED_THRESHOLD or speed2 < SPEED_THRESHOLD:
        moving_vel = vel1 if speed1 > speed2 else vel2
        stationary_bbox = get_bbox_at_frame(
            track2 if speed1 > speed2 else track1, acc_frame)
        if stationary_bbox is not None:
            moving_angle = math.atan2(moving_vel[1], moving_vel[0])
            return "t-bone" if abs(math.sin(moving_angle)) > 0.7 else "rear-end"
        return "rear-end"

    dot = vel1[0]*vel2[0] + vel1[1]*vel2[1]
    cos_angle = np.clip(dot / (speed1 * speed2 + 1e-8), -1.0, 1.0)
    relative_angle = math.degrees(math.acos(cos_angle))

    heading1 = math.atan2(vel1[1], vel1[0])
    heading2 = math.atan2(vel2[1], vel2[0])
    heading_diff = abs(math.degrees(heading1 - heading2)) % 360
    if heading_diff > 180:
        heading_diff = 360 - heading_diff

    if relative_angle > 135:
        return "head-on"
    elif relative_angle < 30:
        return "rear-end"
    elif 60 <= relative_angle <= 120:
        return "t-bone"
    elif 30 <= relative_angle < 60:
        return "sideswipe" if heading_diff < 45 else "rear-end"
    elif 120 < relative_angle <= 135:
        return "head-on"
    else:
        return "rear-end"

# ═══════════════════════════════════════════════════════════
# §7  TriDet temporal inference
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def tridet_predict_time(feats, stamps):
    x = torch.tensor(feats, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    conf, dt = tridet(x)
    conf = conf.squeeze(0).cpu().numpy()
    dt   = dt.squeeze(0).cpu().numpy()

    if conf.max() < 0.1:
        return float(stamps[len(stamps)//2])

    k = max(1, min(5, len(conf)))
    top_idx = np.argsort(conf)[-k:]
    predicted_times = []
    weights = []
    for i in top_idx:
        t_start_pred = stamps[i] - dt[i, 0]
        t_end_pred   = stamps[i] + dt[i, 1]
        t_center     = (t_start_pred + t_end_pred) / 2.0
        predicted_times.append(t_center)
        weights.append(conf[i])
    weights = np.array(weights)
    weights = weights / (weights.sum() + 1e-9)
    return float(np.dot(predicted_times, weights))

# ═══════════════════════════════════════════════════════════
# §8  Fallback peak-based temporal estimation
# ═══════════════════════════════════════════════════════════
def fallback_peak_time(scores, fps):
    if not scores:
        return 0.0
    vals = np.array([s[1] for s in scores], dtype=np.float32)
    k = min(7, len(vals))
    if k >= 3:
        sm = np.convolve(vals, np.ones(k)/k, mode="same")
    else:
        sm = vals
    best = int(np.argmax(sm))
    return float(scores[best][0] / fps)

# ═══════════════════════════════════════════════════════════
# §9  Full per-video inference
# ═══════════════════════════════════════════════════════════
ACCIDENT_FALLBACK = "rear-end"

def infer_one(video_path):
    frames, fps = load_frames_limited(video_path, MAX_VIDEO_SEC)
    if len(frames) == 0:
        return 0.0, 0.5, 0.5, ACCIDENT_FALLBACK
    H, W = frames[0].shape[:2]

    # Step 1: Detection + Tracking
    det, trajectories = run_tracking(frames, fps, detect_fps=DETECT_FPS)

    # Step 2: Interaction scores → candidate windows
    int_scores = compute_interaction_scores(det, trajectories, fps)
    candidates = pick_top_candidates(int_scores, fps, n_candidates=3, min_gap_sec=2.0)

    # Step 3: Temporal localization (features ONLY around candidates)
    if tridet_trained and len(frames) >= 16:
        cand_frames = [c[0] for c in candidates] if candidates else None
        feats, stamps = build_feature_sequence(
            frames, fps, det, trajectories,
            clip_stride_sec=1.0, candidate_frames=cand_frames)
        accident_time = tridet_predict_time(feats, stamps)
        accident_time = max(0.0, min(accident_time, len(frames)/fps))
    else:
        accident_time = fallback_peak_time(int_scores, fps)
        if candidates:
            accident_time = candidates[0][0] / fps

    # Step 4: Spatial localization (impact point)
    acc_frame = int(accident_time * fps)
    acc_frame = max(0, min(acc_frame, len(frames) - 1))

    colliding_pair = find_colliding_pair(det, trajectories, acc_frame, fps)
    ip = np.array([W * 0.5, H * 0.5], dtype=np.float32)

    if colliding_pair is not None:
        tid1, tid2 = colliding_pair
        bb1 = get_bbox_at_frame(trajectories.get(tid1), acc_frame)
        bb2 = get_bbox_at_frame(trajectories.get(tid2), acc_frame)
        if bb1 is not None and bb2 is not None:
            ip = impact_point_from_boxes(bb1, bb2)

    cx = float(np.clip(ip[0] / W, 0.0, 1.0))
    cy = float(np.clip(ip[1] / H, 0.0, 1.0))

    # Step 5: Classification
    acc_type = classify_from_trajectories(det, trajectories, accident_time, fps)

    return accident_time, cx, cy, acc_type

# ═══════════════════════════════════════════════════════════
# §10  Run all test videos + write submission
# ═══════════════════════════════════════════════════════════
rows = []
for vp in tqdm(TEST_VIDEOS, desc="Inference (VideoMAE+TriDet)"):
    try:
        t, cx, cy, tp = infer_one(vp)
    except Exception as e:
        print(f"[ERR] {os.path.basename(vp)}: {e}")
        t, cx, cy, tp = 0.0, 0.5, 0.5, ACCIDENT_FALLBACK

    rows.append({
        "path":          "videos/" + os.path.basename(vp),
        "accident_time": round(t, 3),
        "center_x":      round(cx, 4),
        "center_y":      round(cy, 4),
        "type":          tp,
    })

sub = pd.DataFrame(rows, columns=["path","accident_time","center_x","center_y","type"])
out_path = "/kaggle/working/submission.csv"
sub.to_csv(out_path, index=False)
print("[SAVED]", out_path)
print(sub.head(10))

[OK] #test videos: 2027
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.9 MB/s eta 0:00:00a 0:00:01
[OK] DEVICE: cuda


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/377M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/184 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.v_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.weight           | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.weight              | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.query.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.weight          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.bias                | UNEXPECTED |  | 
decoder.head.weight                                               

[OK] TriDet trained on 60 synth videos, avg loss = 0.6737


Inference (VideoMAE+TriDet):   5%|▌         | 102/2027 [31:03<7:23:59, 13.84s/it]